In [1]:
import torch
from esmfold import ESMFold

In [5]:
pretrained = torch.load("esmfold.model", weights_only=False)
model = ESMFold(esmfold_config=pretrained.cfg)
model.load_state_dict(pretrained.state_dict())

model.eval().cuda().requires_grad_(False)

OutOfMemoryError: CUDA out of memory. Tried to allocate 14.00 MiB. GPU 0 has a total capacity of 23.52 GiB of which 1.06 MiB is free. Process 3722522 has 8.60 GiB memory in use. Process 3856783 has 8.73 GiB memory in use. Process 3913800 has 6.17 GiB memory in use. Of the allocated memory 5.56 GiB is allocated by PyTorch, and 232.07 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
torch.cuda.empty_cache()
output = model.infer(sequence,
                     num_recycles=num_recycles,
                     chain_linker="X"*chain_linker,
                     residue_index_offset=512)

pdb_str = model.output_to_pdb(output)[0]
output = tree_map(lambda x: x.cpu().numpy(), output)
ptm = output["ptm"][0]
plddt = output["plddt"][0,...,1].mean()
O = parse_output(output)
print(f'ptm: {ptm:.3f} plddt: {plddt:.3f}')
os.system(f"mkdir -p {ID}")
prefix = f"{ID}/ptm{ptm:.3f}_r{num_recycles}_default"
np.savetxt(f"{prefix}.pae.txt",O["pae"],"%.3f")
with open(f"{prefix}.pdb","w") as out:
    out.write(pdb_str)